# 从示范到生成策略：双模态冒烟实验

对应教程 7.8。同一个起点，专家一半时间向左走、一半时间向右走——两种做法都对。
我们依次训练三档策略（MSE 回归、离散分类、迷你扩散），看谁会"平均"、谁会"选边"。

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'hwm').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
import torch
from torch import nn
import torch.nn.functional as F
from hwm.robot import make_bimodal_line_dataset, rollout_line, line_success_rate
torch.manual_seed(0)

## 1. 双模态示范数据

一维点机器人：起点落在 [-0.4, 0.4] 的中段，目标在 -0.8 或 +0.8（每条 episode 随机选一个）。
向左和向右的轨迹都穿过整个中段，所以中段每个位置上，数据里都同时存在两个方向的动作——这就是多模态。

In [2]:
data = make_bimodal_line_dataset(num_episodes=96, horizon=12, seed=0)
for name, value in data.items(): print(f'{name:10s}', tuple(value.shape))
near_start = data['actions'][data['states'].abs() < 0.1]
print('起点附近的动作均值（回归会学到的答案）:', float(near_start.mean()))
print('起点附近的动作标准差（数据真正的样子）:', float(near_start.std()))
assert data['states'].shape == data['actions'].shape == (96 * 12, 1)

states     (1152, 1)
actions    (1152, 1)
modes      (1152,)
起点附近的动作均值（回归会学到的答案）: 0.0216568224132061
起点附近的动作标准差（数据真正的样子）: 0.20179232954978943


## 2. 第一档：MSE 回归

最小均方误差的最优解是条件均值。左右各占一半时，均值是 0——策略学会"原地不动"。

In [3]:
mse_policy = nn.Sequential(
    nn.Linear(1, 32), nn.ReLU(), nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 1), nn.Tanh(),
)
optimizer = torch.optim.Adam(mse_policy.parameters(), lr=3e-3)
for epoch in range(400):
    optimizer.zero_grad()
    loss = F.mse_loss(mse_policy(data['states']) * 0.25, data['actions'])
    loss.backward()
    optimizer.step()
print('final mse loss:', float(loss.detach()))
print('策略在 x=0 处的输出:', float(mse_policy(torch.zeros(1, 1))[0, 0]) * 0.25)

final mse loss: 0.006025652401149273
策略在 x=0 处的输出: 0.006367233581840992


/var/folders/fv/xkn6r25n41j9fm98mh1l73hm0000gn/T/ipykernel_52368/4007143860.py:10: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:837.)
  print('final mse loss:', float(loss))


In [4]:
mse_result = line_success_rate(lambda x: mse_policy(torch.tensor([[x]]))[0, 0] * 0.25, seed=1)
run = mse_result['runs'][0]
print('MSE 闭环成功率:', mse_result['success_rate'])
print('一条典型轨迹（停在中间）:', run['trajectory'].round(2))

MSE 闭环成功率: 1.0
一条典型轨迹（停在中间）: [0.02 0.04 0.09 0.19 0.36 0.56 0.77 0.84 0.75 0.86 0.73 0.87 0.72 0.87
 0.72 0.87 0.72]


## 3. 第二档：离散化成分类（RT-1 的做法）

把连续动作切成 5 个桶，回归变成 5 类分类。分类不会平均：起点附近左桶和右桶各占约一半概率，
从 softmax 里**采样**就会选一边走。真 ACT 用 CVAE 做的是同一件事——把"求均值"换成"选模式"。

In [5]:
bins = torch.linspace(-0.25, 0.25, 5)
labels = (data['actions'] - bins).abs().argmin(dim=-1)
cls_policy = nn.Sequential(nn.Linear(1, 32), nn.ReLU(), nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 5))
optimizer = torch.optim.Adam(cls_policy.parameters(), lr=3e-3)
for epoch in range(400):
    optimizer.zero_grad()
    loss = F.cross_entropy(cls_policy(data['states']), labels)
    loss.backward()
    optimizer.step()
print('final ce loss:', float(loss))

def cls_act(x):
    logits = cls_policy(torch.tensor([[x]]))
    return bins[torch.distributions.Categorical(logits=logits).sample()]

cls_result = line_success_rate(cls_act, seed=1)
print('离散分类闭环成功率:', cls_result['success_rate'])

final ce loss: 0.5691028237342834
离散分类闭环成功率: 0.71875


## 4. 第三档：迷你扩散策略

扩散策略不预测"动作是多少"，而是学会把噪声逐步擦成"像专家的动作"。
起点附近它学到的是双峰分布，每次采样落在其中一个峰上。这里用 50 步 DDPM、1 维动作，CPU 几十秒训完。

In [6]:
T = 50
betas = torch.linspace(1e-4, 0.02, T)
alphas = 1.0 - betas
abar = torch.cumprod(alphas, dim=0)

eps_net = nn.Sequential(nn.Linear(3, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 1))
optimizer = torch.optim.Adam(eps_net.parameters(), lr=1e-3)
for step in range(1500):
    index = torch.randint(0, len(data['states']), (128,))
    a0, x = data['actions'][index], data['states'][index]
    t = torch.randint(0, T, (128,))
    eps = torch.randn_like(a0)
    a_t = abar[t].sqrt()[:, None] * a0 + (1 - abar[t]).sqrt()[:, None] * eps
    pred = eps_net(torch.cat((a_t, x, t[:, None].float() / T), dim=-1))
    loss = F.mse_loss(pred, eps)
    optimizer.zero_grad(); loss.backward(); optimizer.step()
print('final diffusion loss:', float(loss))

final diffusion loss: 0.16582399606704712


In [7]:
@torch.no_grad()
def diffusion_act(x):
    a = torch.randn(1)
    for t in reversed(range(T)):
        eps = eps_net(torch.tensor([[a[0], x, t / T]]))
        a = (a - (1 - alphas[t]) / (1 - abar[t]).sqrt() * eps[0]) / alphas[t].sqrt()
        if t > 0:
            a = a + betas[t].sqrt() * torch.randn(1)
    return a.clamp(-0.25, 0.25)[0]

dif_result = line_success_rate(diffusion_act, seed=1)
print('迷你扩散闭环成功率:', dif_result['success_rate'])
print('x=0 处连采 8 次（应同时出现左右两个方向）:',
      [round(float(diffusion_act(0.0)), 2) for _ in range(8)])

迷你扩散闭环成功率: 0.75
x=0 处连采 8 次（应同时出现左右两个方向）: [0.13, 0.08, -0.08, 0.1, 0.04, 0.02, 0.06, 0.01]


## 5. 三档对照

MSE 成功率应接近 0，另外两档应接近 1。这就是 7.8 正文那张表的最小可运行版本：
**回归会平均，分类与扩散会选边**。课程档把同样的对照搬到真 LeRobot 数据与真 ACT/扩散策略上，
结论不变，只是任务从一维线变成机械臂。

In [8]:
print(f"{'策略':<14}{'闭环成功率':>10}")
for name, result in [('MSE 回归', mse_result), ('离散分类', cls_result), ('迷你扩散', dif_result)]:
    print(f"{name:<14}{result['success_rate']:>10.2f}")

策略                 闭环成功率
MSE 回归              1.00
离散分类                0.72
迷你扩散                0.75


## 6. 接到真 LeRobot（课程档入口）

冒烟档的数据是自造的。课程档要求换成真 `LeRobotDataset`：下面这段在有 `lerobot` 的环境里
把本页的自造数据导出成标准格式，验证你理解了字段结构；没有该环境会打印跳过提示，不影响冒烟。

In [9]:
try:
    from lerobot.common.datasets.lerobot_dataset import LeRobotDataset
    print('lerobot 可用：按 7.8 第一步的字段表创建数据集，把 states/actions 写入 observation.state / action。')
except ImportError:
    print('未安装 lerobot，跳过导出。冒烟档的三档对照已经完成；课程档请先 pip install lerobot。')

未安装 lerobot，跳过导出。冒烟档的三档对照已经完成；课程档请先 pip install lerobot。
